## Challenge three: Gemini + BigQuery: Enhancing Public Communication

In [1]:
from google.cloud import bigquery

### variables

In [2]:
import os

# --- Diagnostic: shows what each detection method returns ---
print("=== Project detection diagnostic ===")
print("env GOOGLE_CLOUD_PROJECT:", os.environ.get("GOOGLE_CLOUD_PROJECT"))
print("env GCP_PROJECT         :", os.environ.get("GCP_PROJECT"))
try:
    import google.auth
    _creds, _proj = google.auth.default()
    print("google.auth project     :", _proj)
except Exception as e:
    print("google.auth failed      :", e)
print("=" * 36)


def detect_project_id():
    # 1. Explicit env var wins (lets anyone override without editing code)
    env = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT")
    if env:
        return env
    # 2. Ask Application Default Credentials what project we're running under
    try:
        import google.auth
        _, project = google.auth.default()
        if project:
            return project
    except Exception:
        pass
    # 3. Last resort: query the metadata server (works on GCP runtimes)
    try:
        import urllib.request
        req = urllib.request.Request(
            "http://metadata.google.internal/computeMetadata/v1/project/project-id",
            headers={"Metadata-Flavor": "Google"},
        )
        return urllib.request.urlopen(req, timeout=2).read().decode()
    except Exception:
        return None


class Config:
    def __init__(
        self,
        project_id=None,                      # None -> auto-detect from the runtime
        dataset="weather_comms",
        location="US",                        # must match the GCS bucket region
        source_uri="gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv",
        raw_table_name="weather_data",
        report_table_name="weather_data_with_reports",
        connection_name="gemini_conn",
        model_name="gemini_model",
        endpoint="gemini-2.5-flash",
    ):
        resolved = project_id or detect_project_id()
        if not resolved:
            raise ValueError(
                "Could not determine project_id. Pass it explicitly: "
                "Config(project_id='my-project')"
            )
        # bypass our own __setattr__ block during construction
        object.__setattr__(self, "project_id", resolved)
        object.__setattr__(self, "dataset", dataset)
        object.__setattr__(self, "location", location)
        object.__setattr__(self, "source_uri", source_uri)
        object.__setattr__(self, "raw_table_name", raw_table_name)
        object.__setattr__(self, "report_table_name", report_table_name)
        object.__setattr__(self, "connection_name", connection_name)
        object.__setattr__(self, "model_name", model_name)
        object.__setattr__(self, "endpoint", endpoint)

    def __setattr__(self, name, value):
        raise AttributeError(f"Config is immutable; can't set {name!r}")

    @property
    def raw_table(self):
        return f"{self.project_id}.{self.dataset}.{self.raw_table_name}"

    @property
    def report_table(self):
        return f"{self.project_id}.{self.dataset}.{self.report_table_name}"

    @property
    def model(self):
        return f"{self.project_id}.{self.dataset}.{self.model_name}"

    @property
    def connection(self):
        return f"{self.project_id}.{self.location}.{self.connection_name}"


# --- Build the config ---
# Auto-detects on a GCP runtime. If detection fails, either:
#   - set os.environ["GOOGLE_CLOUD_PROJECT"] = "your-project" above, OR
#   - pass it directly: Config(project_id="your-project")
CFG = Config()

print("\nProject    :", CFG.project_id)
print("Raw table  :", CFG.raw_table)
print("Report tbl :", CFG.report_table)
print("Connection :", CFG.connection)
print("Model      :", CFG.model)

=== Project detection diagnostic ===
env GOOGLE_CLOUD_PROJECT: qwiklabs-gcp-01-5fe45b5e4e14
env GCP_PROJECT         : None
google.auth project     : qwiklabs-gcp-01-5fe45b5e4e14

Project    : qwiklabs-gcp-01-5fe45b5e4e14
Raw table  : qwiklabs-gcp-01-5fe45b5e4e14.weather_comms.weather_data
Report tbl : qwiklabs-gcp-01-5fe45b5e4e14.weather_comms.weather_data_with_reports
Connection : qwiklabs-gcp-01-5fe45b5e4e14.US.gemini_conn
Model      : qwiklabs-gcp-01-5fe45b5e4e14.weather_comms.gemini_model


### Logging

In [3]:
import logging, json, sys, uuid, datetime as dt

logger = logging.getLogger("weather_comms_ml")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(h)

RUN_ID = str(uuid.uuid4())

def log_event(step: str, status: str, **kw):
    logger.info(json.dumps({"run_id": RUN_ID, "step": step, "status": status, **kw}))

log_event("init", "ok", started=dt.datetime.now(dt.timezone.utc).isoformat())

2026-06-02 19:05:29,082 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "init", "status": "ok", "started": "2026-06-02T19:05:29.082408+00:00"}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "init", "status": "ok", "started": "2026-06-02T19:05:29.082408+00:00"}


### Pipeline: Gemini-powered weather communications

In [4]:
import subprocess, time


class WeatherCommsPipeline:
    """Loads raw weather data, registers Gemini as a remote model in BigQuery,
    and uses ML.GENERATE_TEXT to write a public weather report for every row."""

    # Registers Gemini as a BigQuery ML remote model via a CLOUD_RESOURCE connection.
    CREATE_MODEL_SQL = """
    CREATE OR REPLACE MODEL `{model}`
    REMOTE WITH CONNECTION `{connection}`
    OPTIONS (ENDPOINT = '{endpoint}')
    """

    # Builds a per-row prompt from the weather fields, calls Gemini inside SQL,
    # and writes the LLM response into a new table as `weather_report`.
    GENERATE_SQL = """
    CREATE OR REPLACE TABLE `{report_table}` AS
    SELECT
      * EXCEPT(prompt, ml_generate_text_rai_result, ml_generate_text_status),
      ml_generate_text_llm_result AS weather_report
    FROM ML.GENERATE_TEXT(
      MODEL `{model}`,
      (
        SELECT
          *,
          CONCAT(
            'You are a public safety communications officer. ',
            'Write a short, clear weather report or warning for the public ',
            'based on the following conditions. Keep it under 60 words. ',
            'City: ', city, ', ', state,
            '. Date: ', CAST(date AS STRING),
            '. Temperature: ', CAST(temperature_f AS STRING), ' F',
            '. Condition: ', weather_condition,
            '. Wind speed: ', CAST(wind_speed_mph AS STRING), ' mph',
            '. Precipitation: ', CAST(precipitation_in AS STRING), ' in',
            '. Humidity: ', CAST(humidity_percent AS STRING), '%',
            '. Barometric pressure: ', CAST(barometric_pressure_inHg AS STRING), ' inHg'
          ) AS prompt
        FROM `{raw_table}`
      ),
      STRUCT(
        0.3   AS temperature,          -- low = consistent, factual tone
        1024  AS max_output_tokens,
        TRUE  AS flatten_json_output   -- clean `ml_generate_text_llm_result` column
      )
    )
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.client = bigquery.Client(project=cfg.project_id, location=cfg.location)
        self.rows_in = 0
        self.service_account = None

    def ensure_dataset(self):
        ds = bigquery.Dataset(f"{self.cfg.project_id}.{self.cfg.dataset}")
        ds.location = self.cfg.location
        self.client.create_dataset(ds, exists_ok=True)
        log_event("dataset_ready", "ok", dataset=self.cfg.dataset)

    def load_raw(self):
        job_config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,
            skip_leading_rows=1,
            autodetect=True,
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        )
        job = self.client.load_table_from_uri(
            self.cfg.source_uri, self.cfg.raw_table, job_config=job_config
        )
        job.result()
        self.rows_in = self.client.get_table(self.cfg.raw_table).num_rows
        log_event("load_raw", "ok", rows=self.rows_in, job_id=job.job_id)

    def preview_raw(self, n: int = 10):
        df = self.client.query(
            f"SELECT * FROM `{self.cfg.raw_table}` LIMIT {n}"
        ).to_dataframe()
        log_event("preview_raw", "ok", rows=len(df))
        return df

    def schema(self):
        table = self.client.get_table(self.cfg.raw_table)
        df = self.client.query(f"""
        SELECT column_name, data_type
        FROM `{self.cfg.project_id}.{self.cfg.dataset}.INFORMATION_SCHEMA.COLUMNS`
        WHERE table_name = '{self.cfg.raw_table_name}'
        ORDER BY ordinal_position
        """).to_dataframe()
        log_event("schema", "ok", columns=len(df))
        return df

    def profile(self):
        df = self.client.query(f"""
        SELECT
          COUNT(*) AS row_count,
          COUNT(DISTINCT city) AS cities,
          COUNT(DISTINCT weather_condition) AS conditions,
          ROUND(AVG(temperature_f), 2) AS avg_temp_f,
          ROUND(MAX(wind_speed_mph), 2) AS max_wind_mph,
          ROUND(MAX(precipitation_in), 2) AS max_precip_in
        FROM `{self.cfg.raw_table}`
        """).to_dataframe()
        log_event("profile", "ok")
        return df

    def conditions_breakdown(self):
        df = self.client.query(f"""
        SELECT
          weather_condition,
          COUNT(*) AS days,
          ROUND(AVG(temperature_f), 1) AS avg_temp_f,
          ROUND(AVG(wind_speed_mph), 1) AS avg_wind_mph
        FROM `{self.cfg.raw_table}`
        GROUP BY weather_condition
        ORDER BY days DESC
        """).to_dataframe()
        log_event("conditions_breakdown", "ok", groups=len(df))
        return df

    def ensure_connection(self):
        """Create the CLOUD_RESOURCE connection and capture its service account."""
        subprocess.run(
            ["bq", "mk", "--connection", f"--location={self.cfg.location}",
             f"--project_id={self.cfg.project_id}",
             "--connection_type=CLOUD_RESOURCE", self.cfg.connection_name],
            capture_output=True, text=True,
        )  # ignore "already exists"
        info = subprocess.run(
            ["bq", "show", "--format=json", "--connection", self.cfg.connection],
            capture_output=True, text=True,
        )
        self.service_account = json.loads(info.stdout)["cloudResource"]["serviceAccountId"]
        log_event("ensure_connection", "ok", service_account=self.service_account)
        return self.service_account

    def grant_vertex_role(self, wait_seconds: int = 60):
        """Grant the connection SA the aiplatform.user role and let it propagate."""
        subprocess.run(
            ["gcloud", "projects", "add-iam-policy-binding", self.cfg.project_id,
             f"--member=serviceAccount:{self.service_account}",
             "--role=roles/aiplatform.user", "--condition=None"],
            capture_output=True, text=True,
        )
        log_event("grant_vertex_role", "ok", waiting=wait_seconds)
        time.sleep(wait_seconds)

    def create_model(self):
        sql = self.CREATE_MODEL_SQL.format(
            model=self.cfg.model, connection=self.cfg.connection,
            endpoint=self.cfg.endpoint,
        )
        job = self.client.query(sql)
        job.result()
        log_event("create_model", "ok", model=self.cfg.model_name,
                  endpoint=self.cfg.endpoint, job_id=job.job_id)

    def generate_reports(self):
        sql = self.GENERATE_SQL.format(
            report_table=self.cfg.report_table, model=self.cfg.model,
            raw_table=self.cfg.raw_table,
        )
        job = self.client.query(sql)
        job.result()
        rows = self.client.get_table(self.cfg.report_table).num_rows
        log_event("generate_reports", "ok", rows=rows, job_id=job.job_id)

    def preview_reports(self, n: int = 10):
        df = self.client.query(f"""
        SELECT date, city, state, weather_condition, weather_report
        FROM `{self.cfg.report_table}`
        LIMIT {n}
        """).to_dataframe()
        log_event("preview_reports", "ok", rows=len(df))
        return df

    def run(self):
        self.ensure_dataset()
        self.load_raw()
        self.ensure_connection()
        self.grant_vertex_role()
        self.create_model()
        self.generate_reports()
        log_event("run", "ok", rows_in=self.rows_in)

### Setup: dataset + load raw data

In [5]:
pipeline = WeatherCommsPipeline(CFG)
pipeline.ensure_dataset()
pipeline.load_raw()
print(f"Loaded {pipeline.rows_in:,} rows into {CFG.raw_table}")

2026-06-02 19:07:28,951 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "dataset_ready", "status": "ok", "dataset": "weather_comms"}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "dataset_ready", "status": "ok", "dataset": "weather_comms"}


2026-06-02 19:07:34,589 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "load_raw", "status": "ok", "rows": 300, "job_id": "ed58c85d-fa3f-435d-9c7c-c35191184b04"}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "load_raw", "status": "ok", "rows": 300, "job_id": "ed58c85d-fa3f-435d-9c7c-c35191184b04"}


Loaded 300 rows into qwiklabs-gcp-01-5fe45b5e4e14.weather_comms.weather_data


### Preview the raw data

In [6]:
pipeline.preview_raw()

2026-06-02 19:08:09,448 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "preview_raw", "status": "ok", "rows": 10}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "preview_raw", "status": "ok", "rows": 10}


,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition
0,2025-02-21,Atlanta,GA,55.7,5.0,0.12,29.80,50.4,Cloudy
1,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy
2,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy
3,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy
4,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy
5,2025-03-14,Atlanta,GA,71.7,7.2,0.18,29.92,55.3,Cloudy
6,2025-02-19,Boston,MA,61.7,3.9,0.11,29.62,54.1,Cloudy
7,2025-03-09,Boston,MA,76.7,4.3,0.09,29.52,40.9,Cloudy
8,2025-03-13,Boston,MA,71.9,9.8,0.16,29.99,42.3,Cloudy
9,2025-03-19,Boston,MA,60.7,6.4,0.04,29.83,49.4,Cloudy


### Study the data — schema

In [7]:
pipeline.schema()

2026-06-02 19:08:43,645 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "schema", "status": "ok", "columns": 9}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "schema", "status": "ok", "columns": 9}


,column_name,data_type
0,date,DATE
1,city,STRING
2,state,STRING
3,temperature_f,FLOAT64
4,wind_speed_mph,FLOAT64
5,precipitation_in,FLOAT64
6,barometric_pressure_inHg,FLOAT64
7,humidity_percent,FLOAT64
8,weather_condition,STRING


### Profile the weather data

In [8]:
pipeline.profile()

2026-06-02 19:09:18,605 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "profile", "status": "ok"}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "profile", "status": "ok"}


,row_count,cities,conditions,avg_temp_f,max_wind_mph,max_precip_in
0,300,10,5,53.42,49.8,3.9


### Breakdown by weather condition

In [9]:
pipeline.conditions_breakdown()

2026-06-02 19:09:55,127 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "conditions_breakdown", "status": "ok", "groups": 5}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "conditions_breakdown", "status": "ok", "groups": 5}


,weather_condition,days,avg_temp_f,avg_wind_mph
0,Stormy,72,43.9,34.8
1,Snowy,62,27.2,11.4
2,Sunny,60,81.4,2.8
3,Cloudy,55,63.2,7.2
4,Rainy,51,55.2,11.7


### Connect to Vertex AI — create connection + grant role

In [10]:
pipeline.ensure_connection()
print("Connection service account:", pipeline.service_account)
pipeline.grant_vertex_role()
print("IAM role granted and propagated.")

2026-06-02 19:10:27,384 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "ensure_connection", "status": "ok", "service_account": "bqcx-299593570351-o8m6@gcp-sa-bigquery-condel.iam.gserviceaccount.com"}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "ensure_connection", "status": "ok", "service_account": "bqcx-299593570351-o8m6@gcp-sa-bigquery-condel.iam.gserviceaccount.com"}


Connection service account: bqcx-299593570351-o8m6@gcp-sa-bigquery-condel.iam.gserviceaccount.com
2026-06-02 19:10:28,736 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "grant_vertex_role", "status": "ok", "waiting": 60}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "grant_vertex_role", "status": "ok", "waiting": 60}


IAM role granted and propagated.


### Add Gemini to the project — CREATE MODEL

In [11]:
pipeline.create_model()
print(f"Remote model {CFG.model} created, pointing to {CFG.endpoint}.")

2026-06-02 19:11:40,833 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "create_model", "status": "ok", "model": "gemini_model", "endpoint": "gemini-2.5-flash", "job_id": "36f26c66-5a2e-4520-b450-4db69bc6d725"}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "create_model", "status": "ok", "model": "gemini_model", "endpoint": "gemini-2.5-flash", "job_id": "36f26c66-5a2e-4520-b450-4db69bc6d725"}


Remote model qwiklabs-gcp-01-5fe45b5e4e14.weather_comms.gemini_model created, pointing to gemini-2.5-flash.


### Generate weather reports in SQL — ML.GENERATE_TEXT

In [12]:
pipeline.generate_reports()
print(f"Weather reports written to {CFG.report_table}.")

2026-06-02 19:12:29,257 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "generate_reports", "status": "ok", "rows": 300, "job_id": "8fc8fdd0-6273-4d36-a7b9-b0e1451b6f99"}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "generate_reports", "status": "ok", "rows": 300, "job_id": "8fc8fdd0-6273-4d36-a7b9-b0e1451b6f99"}


Weather reports written to qwiklabs-gcp-01-5fe45b5e4e14.weather_comms.weather_data_with_reports.


### Review the generated reports

In [13]:
pipeline.preview_reports()

2026-06-02 19:12:55,543 | INFO | {"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "preview_reports", "status": "ok", "rows": 10}


INFO:weather_comms_ml:{"run_id": "57e232ff-21f3-49fb-b513-d5afa7becd35", "step": "preview_reports", "status": "ok", "rows": 10}


,date,city,state,weather_condition,weather_report
0,2025-03-01,Atlanta,GA,Cloudy,Atlanta: March 1st. Cloudy with light rain. Te...
1,2025-02-26,Atlanta,GA,Cloudy,Atlanta: Feb 26. Cloudy with light rain (0.03 ...
2,2025-03-10,Atlanta,GA,Cloudy,Atlanta: March 10th. Cloudy with light rain. T...
3,2025-03-05,Atlanta,GA,Cloudy,Atlanta: Cloudy with light rain possible today...
4,2025-02-21,Atlanta,GA,Cloudy,Atlanta: Cloudy with light rain today. Tempera...
5,2025-03-14,Atlanta,GA,Cloudy,"Atlanta, today, March 14th: Cloudy with light ..."
6,2025-03-13,Boston,MA,Cloudy,**Boston Weather Alert:** March 13th. Expect c...
7,2025-03-09,Boston,MA,Cloudy,Boston: March 9th. Cloudy with light rain (0.0...
8,2025-03-19,Boston,MA,Cloudy,"Boston, today, March 19th: Cloudy with light r..."
9,2025-02-19,Boston,MA,Cloudy,Boston: February 19th. Cloudy with light rain ...
